In [1]:
from google.colab import drive
drive.mount('/content/drive/')

# Cambia la directory di lavoro in quella del tuo progetto
# (Assicurati che il percorso sia esattamente quello in cui tieni la cartella eomt sul tuo Drive)
%cd /content/drive/MyDrive/project/semantic-segmentation-roads/eomt

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
/content/drive/.shortcut-targets-by-id/1kGi4cSNJjM14ClVvPXJ9VY70JDkofHGn/project/semantic-segmentation-roads/eomt


In [2]:
!pip install -r requirements.txt

In [3]:
import os, sys, importlib

PROJECT_DIR = "/content/drive/MyDrive/project/semantic-segmentation-roads/eomt"

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
importlib.invalidate_caches()

from my_eval import *

IMG_SIZE = (640,640)

# Assicuriamoci di essere sulla GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# --- 1. CONFIGURAZIONI ---
# Usiamo la configurazione di Cityscapes per l'architettura (così avrà 19 classi)
config = get_config()

# --- 2. INIZIALIZZAZIONE DEL DATASET (Cityscapes) ---
data = get_data(config, batch_size=2, img_size=IMG_SIZE)


state_dict_path_coco = "/content/drive/MyDrive/project/eomt_coco.bin"

# --- 3. INIZIALIZZAZIONE DELL'ARCHITETTURA (Cityscapes) ---
model = get_model(config, img_size=IMG_SIZE, num_classes=data.num_classes, masked_attn_enabled=True)

# --- 4. CARICAMENTO DEI PESI DA COCO ---
model = load_weights(model, state_dict_path_coco, device).to(device)

print("\n✅ Modello e Dati pronti per il fine-tuning!")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


Ignored network.q.weight (shape mismatch or not existing)
Ignored network.class_head.weight (shape mismatch or not existing)
Ignored network.class_head.bias (shape mismatch or not existing)
Ignored criterion.empty_weight (shape mismatch or not existing)
Missing keys: ['network.q.weight', 'network.class_head.weight', 'network.class_head.bias', 'criterion.empty_weight']
Unexpected keys: []

✅ Modello e Dati pronti per il fine-tuning!


Step 2: congelamento pesi

In [4]:
# --- 1. CONGELAMENTO GLOBALE (FREEZING) ---
# Diciamo a PyTorch di non calcolare i gradienti per NESSUN parametro del modello
for param in model.parameters():
    param.requires_grad = False

# --- 2. SCONGELAMENTO DELLA PREDICTION HEAD ---
# Riattiviamo i gradienti SOLO per il layer di classificazione finale
# (Guardando nel codice originale, si chiama esattamente "class_head")
for param in model.network.class_head.parameters():
    param.requires_grad = True

for param in model.network.upscale.parameters():
  param.requires_grad = True

for param in model.network.mask_head.parameters():
  param.requires_grad = True

for param in model.network.q.parameters():
  param.requires_grad = True

# --- 4. VERIFICA ---
# Stampiamo un riepilogo per essere sicuri di non far esplodere la RAM di Colab!
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Parametri TOTALI del modello: {total_params:,}")
print(f"Parametri ADDESTRABILI (scongelati): {trainable_params:,}")
print(f"Percentuale di pesi in addestramento: {(trainable_params/total_params)*100:.4f}%")


Parametri TOTALI del modello: 93,498,644
Parametri ADDESTRABILI (scongelati): 6,600,980
Percentuale di pesi in addestramento: 7.0600%


Step 3: configurare il Trainer e avviare il Training

In [5]:
import os
from datetime import datetime
import torch

max_epochs = 20

# -------------------------
# 0. Nome run e cartelle
# -------------------------
run_name = datetime.now().strftime(f"4_components_{max_epochs}ep")

ckpt_dir = f"/content/drive/MyDrive/project/checkpoints/{run_name}"
os.makedirs(ckpt_dir, exist_ok=True)

old_ckpt_dir = "/content/drive/MyDrive/project/checkpoints/4_components_20ep"
resume_ckpt_path = f"{old_ckpt_dir}/last.ckpt"
print("Checkpoint exists:", os.path.exists(resume_ckpt_path))
print(resume_ckpt_path)



Checkpoint exists: True
/content/drive/MyDrive/project/checkpoints/4_components_20ep/last.ckpt


Una volta che l'addestramento (anche di sole 5 epoche) è terminato, il progetto ti chiede di:

Valutare il modello che hai appena addestrato usando la metrica mIoU.
Confrontarlo con gli altri modelli.
Per fare la valutazione, ti basterà ricopiare esattamente la cella che valuta la mIoU che era presente nel tuo Step 4 originario, ma passandogli questo nuovo model che ora è stato perfezionato su Cityscapes!

In [6]:
import wandb
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import WandbLogger



# 1. Eseguiamo il login a Weights & Biases
# (Ti chiederà la chiave API se non sei già loggata nella sessione corrente)
wandb.login(key='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')

# 2. Creiamo il logger: tutto verrà salvato sul tuo profilo nel progetto "eomt-cityscapes"
wandb_logger = WandbLogger(project="eomt-cityscapes", name=run_name, log_model=False)

# Log config utile su WandB
wandb_logger.experiment.config.update({
    "run_name": run_name,
    "max_epochs": max_epochs,
    # "real_epochs": real_epochs,
    "precision": "16-mixed",
    "batch_size": getattr(data, "batch_size", None),
    "img_size": getattr(data, "img_size", None),
    "num_classes": getattr(data, "num_classes", None),
    "finetuning": run_name,
    # "resume_from": resume_ckpt_path
})

# 3. Salvataggio dei pesi (checkpoint) su Drive
checkpoint_callback = ModelCheckpoint(
    dirpath=ckpt_dir,
    filename="best-{epoch:02d}",
    save_last=True,
    save_top_k=2,
    monitor="metrics/val_iou_all",
    mode="max"
)

# Log learning rate su WandB
lr_monitor = LearningRateMonitor(logging_interval="step")

print(f"Checkpoint directory: {ckpt_dir}")



print("Configurazione del PyTorch Lightning Trainer...")

trainer = pl.Trainer(
    max_epochs=max_epochs,
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    logger=wandb_logger,
    callbacks=[checkpoint_callback, lr_monitor],

    log_every_n_steps=10,

    # Per run seria: sanity check attivo, ma non enorme
    num_sanity_val_steps=2,

    # Utile su Colab per non perdere tutto se crasha
    enable_checkpointing=True,

    # Se vuoi evitare training infinito per bug strani
    enable_progress_bar=True,

    # Ricordati: se vedi che si incastra sul Sanity Checking, ferma tutto,
    # copia i file ZIP in locale su /content/ e togli questi limiti!
    # limit_train_batches=20,
    # limit_val_batches=5,
)

print("🚀 Avvio del Fine-Tuning!")
trainer.fit(model, datamodule=data
            # )
, ckpt_path=resume_ckpt_path)

# Alla fine dell'addestramento, diciamo a WandB che abbiamo finito
wandb.finish()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ai-group-3434 (ai-group-3434-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs


Checkpoint directory: /content/drive/MyDrive/project/checkpoints/4_components_20ep
Configurazione del PyTorch Lightning Trainer...
🚀 Avvio del Fine-Tuning!


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/.shortcut-targets-by-id/1kGi4cSNJjM14ClVvPXJ9VY70JDkofHGn/project/checkpoints/4_components_20ep exists and is not empty.
INFO: Restoring states from the checkpoint path at /content/drive/MyDrive/project/checkpoints/4_components_20ep/last.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at /content/drive/MyDrive/project/checkpoints/4_components_20ep/last.ckpt
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:362: The dirpath has changed from '/content/drive/MyDrive/project/checkpoints/4_components_20ep' to '/content/drive/.shortcut-targets-by-id/1kGi4cSNJjM14ClVvPXJ9VY70JDkofHGn/project/checkpoints/4_components_20ep', therefore `best_model_score`, `kth_best_model_path`, `kth_value`, `last_model_path` and `best_k_models` won't be reloaded. Only `best_model_path` will be reloaded.
INF

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

INFO: mIoU: 72.1
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 72.1


Validation: |          | 0/? [00:00<?, ?it/s]

INFO: mIoU: 72.1
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 72.1
INFO: `Trainer.fit` stopped: `max_epochs=20` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=20` reached.


attn_mask_prob_0,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
attn_mask_prob_1,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
attn_mask_prob_2,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
losses/train_loss_cross_entropy,▃▄█▅▃▄▃▄▃▃▄▂▃▂▂▆▄▂▃▄▂▅▁▂▂▄▅▄▂▄▄▂▂▁▂▄▃▃▂▂
losses/train_loss_cross_entropy_block_-1,▂▄▃▁▂▇▅▁▂▃▄▃▃▃▃▂▄▁▅▆▃▃▂▂▅▃▃▃▂▁▅▂▅█▃▄▃▃▄▅
losses/train_loss_cross_entropy_block_-2,▃▂▃▄▂█▁▂▃▃▃▂▁▂▃▁▁▁▂▃▃▇▃▅▂▃▂▂▃▄▅▃▃▃▁▃▂▁▂▃
losses/train_loss_cross_entropy_block_-3,▁▆▃▁▅▁▄▃▅▂▂▁▆▃▆▅▂▄▄▂▃▅▅▅▄▃▄▃▂█▂▁▁▆▃▅▄▃▁▂
losses/train_loss_dice,▃▃▃▃▆▃▄▂▃▃▄▃▃▆█▅▆▄▄▃▁▆▄▃▅▇▅▄▃▅▂▂▄▄▃▂▃▁▄▄
losses/train_loss_dice_block_-1,▂▅▃█▁▃▄▄▃▄▂▅▃▂▃▃▂▅▃▄▂▅▃▄▅▄▄▄▂▃▄▁▅▄▁▇▂▃▅▅
+206,...
